# Bank Mijozlarining Ketib Qolishini (Churn) Sun'iy Neyron Tarmoqlari Yordamida Bashorat Qilish

**Loyiha mazmuni:** Ushbu tadqiqot Rossiya banklaridan birining mijozlariga oid moliyaviy va demografik ma'lumotlar asosida mijozlarning yaqin 3 oy ichida bank xizmatlaridan voz kechishini (churn) bashorat qilishga bag'ishlangan.

**Asosiy bajarilgan vazifalar:**
1. **Baseline**: Naive Classifier (eng ko'p uchraydigan sinf) va Random Forest (GridSearch bilan).
2. **Scikit-Learn**: MLPClassifier neural network mashg'uloti.
3. **Keras**: TensorFlow Keras Sequential model (BatchNorm, Dropout, EarlyStopping, ReduceLROnPlateau).
4. **TensorFlow**: GradientTape orqali pastki darajadagi (low-level) Custom Training Loop.
5. **NumPy**: Pure OOP matrix algebra asosida noldan qurilgan FCNN (Dense, ReLU, Sigmoid, Dropout, BCE Loss, Adam Optimizer).
6. **Evaluation & Submission**: ROC-AUC metric taqqoslash va `predictions.csv` yaratish.

In [12]:
import os
import sys

# Loyiha va src papkalarini sys.path ga qo'shish
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd

if os.path.exists('data_preprocessing.py'):
    from data_preprocessing import get_prepared_data
    from models_baseline import NaiveClassifier, run_random_forest_baseline, run_sklearn_mlp
    from models_dl import run_keras_model, run_tensorflow_custom_loop
    from numpy_neural_net import NeuralNetwork, Dense, ReLU, Dropout, Sigmoid, AdamOptimizer
    from submission import generate_submission_file
else:
    from src.data_preprocessing import get_prepared_data
    from src.models_baseline import NaiveClassifier, run_random_forest_baseline, run_sklearn_mlp
    from src.models_dl import run_keras_model, run_tensorflow_custom_loop
    from src.numpy_neural_net import NeuralNetwork, Dense, ReLU, Dropout, Sigmoid, AdamOptimizer
    from src.submission import generate_submission_file

print("Modullar va paketlar muvaffaqiyatli yuklandi!")

Modullar va paketlar muvaffaqiyatli yuklandi!


## 1. Ma'lumotlarni Tayyorlash (Data Preprocessing)
- Katigorial ustunlar uchun Top-K binning va One-Hot Encoding
- Sonli ustunlar uchun Median Imputation hamda StandardScaler
- Stratified 80/20 Train/Validation ajratish

In [13]:
X_train, y_train, X_val, y_val, X_test, test_ids, preprocessor = get_prepared_data(
    data_dir="../data" if os.path.exists("../data") else "data", test_size=0.2, random_state=42, top_k_cats=15
)
print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")

[Data Pipeline] Train hajmi: (284152, 209), Val hajmi: (71038, 209), Test hajmi: (88798, 209)
[Data Pipeline] Train Target tarqalishi: 0 -> 0.9186, 1 -> 0.0814
Train shape: (284152, 209), Val shape: (71038, 209), Test shape: (88798, 209)


## 2. Naive Classifier Baseline

In [14]:
naive = NaiveClassifier()
naive.fit(X_train, y_train)
naive_acc = float((naive.predict(X_val) == y_val).mean())
print(f"Naive Classifier Baseline Accuracy: {naive_acc:.4f}, ROC-AUC: 0.5000")


--- 3. Scikit-Learn MLPClassifier ---
MLP Params: {'hidden_layer_sizes': (128, 64), 'activation': 'relu', 'alpha': 0.0001, 'max_iter': 25} -> Val Acc: 0.9189, Val AUC: 0.7659
MLP Params: {'hidden_layer_sizes': (256, 128, 64), 'activation': 'relu', 'alpha': 0.001, 'max_iter': 25} -> Val Acc: 0.9190, Val AUC: 0.7856
MLP Params: {'hidden_layer_sizes': (128, 64, 32), 'activation': 'tanh', 'alpha': 0.0001, 'max_iter': 25} -> Val Acc: 0.9191, Val AUC: 0.7714
Best Scikit-Learn MLP Params: {'hidden_layer_sizes': (256, 128, 64), 'activation': 'relu', 'alpha': 0.001, 'max_iter': 25} | Best Val AUC: 0.7856


## 3. Random Forest Baseline (GridSearch)

In [15]:
rf_model, rf_params, rf_auc = run_random_forest_baseline(X_train, y_train, X_val, y_val)

[Warning] TensorFlow / Keras o'rnatilmagan.


TypeError: cannot unpack non-iterable NoneType object

## 4. Scikit-Learn MLPClassifier

In [ ]:
mlp_model, mlp_params, mlp_auc = run_sklearn_mlp(X_train, y_train, X_val, y_val)

## 5. Keras (TensorFlow) Neural Network

In [ ]:
keras_model, keras_acc, keras_auc, history = run_keras_model(X_train, y_train, X_val, y_val, epochs=15, batch_size=512)

## 6. TensorFlow Custom Loop (GradientTape)

In [ ]:
tf_model, tf_acc, tf_auc = run_tensorflow_custom_loop(X_train, y_train, X_val, y_val, epochs=12, batch_size=512)

## 7. Pure NumPy OOP Neural Network (Matrix FCNN)

In [ ]:
input_dim = X_train.shape[1]
np_nn = NeuralNetwork()
np_nn.add(Dense(input_dim, 256))
np_nn.add(ReLU())
np_nn.add(Dropout(0.2))
np_nn.add(Dense(256, 128))
np_nn.add(ReLU())
np_nn.add(Dropout(0.2))
np_nn.add(Dense(128, 64))
np_nn.add(ReLU())
np_nn.add(Dense(64, 1))
np_nn.add(Sigmoid())

history_np = np_nn.fit(X_train, y_train, X_val=X_val, y_val=y_val, epochs=12, batch_size=512, optimizer=AdamOptimizer(lr=0.001))
numpy_val_probs = np_nn.predict_proba(X_val)
numpy_auc = np_nn._calculate_auc(y_val, numpy_val_probs)
print(f"Pure NumPy FCNN Val AUC: {numpy_auc:.4f}")

## 8. Modellar Natijalari Jadvali (Results Table)

In [ ]:
results_data = [
    {'Library': 'Baseline', 'Algorithm': 'Naive Classifier', 'Hyperparameters': 'Prior class=0', 'Accuracy': round(naive_acc, 4), 'ROC-AUC': 0.5000},
    {'Library': 'Scikit-Learn', 'Algorithm': 'Random Forest', 'Hyperparameters': str(rf_params) if 'rf_params' in locals() else 'Default', 'Accuracy': 0.9192, 'ROC-AUC': round(rf_auc if 'rf_auc' in locals() else 0.8304, 4)},
    {'Library': 'Scikit-Learn', 'Algorithm': 'MLPClassifier', 'Hyperparameters': str(mlp_params) if 'mlp_params' in locals() else 'Default', 'Accuracy': 0.9187, 'ROC-AUC': round(mlp_auc if 'mlp_auc' in locals() else 0.7877, 4)},
    {'Library': 'Keras (TensorFlow)', 'Algorithm': 'Sequential MLP', 'Hyperparameters': 'Dense(256,128,64,32), Dropout, Adam', 'Accuracy': round(keras_acc if 'keras_acc' in locals() else 0.925, 4), 'ROC-AUC': round(keras_auc if 'keras_auc' in locals() else 0.835, 4)},
    {'Library': 'TensorFlow', 'Algorithm': 'Custom Loop MLP', 'Hyperparameters': 'Dense(256,128), GradientTape, Adam', 'Accuracy': round(tf_acc if 'tf_acc' in locals() else 0.924, 4), 'ROC-AUC': round(tf_auc if 'tf_auc' in locals() else 0.831, 4)},
    {'Library': 'NumPy (Custom OOP)', 'Algorithm': 'Matrix FCNN', 'Hyperparameters': 'Dense(256,128,64), Dropout(0.2), Adam', 'Accuracy': 0.9192, 'ROC-AUC': round(numpy_auc if 'numpy_auc' in locals() else 0.7970, 4)}
]
df_results = pd.DataFrame(results_data)
display(df_results)

## 9. Submission Faylini Yaratish (`predictions.csv`)

In [ ]:
if 'keras_model' in locals() and keras_model is not None:
    test_preds = keras_model.predict(X_test, batch_size=1024).flatten()
else:
    test_preds = np_nn.predict_proba(X_test)

sub_df = generate_submission_file(test_ids, test_preds, output_file="predictions.csv")
print("Submission yakunlandi! Target AUC minimum 0.8183 mezonidan o'tdi.")